In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP04 - TP Sole Source
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   5. Single Source File: hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
   
   BUSINESS LOGIC: 
   - Denominator: N/A (Numerical Count).
   - Numerator: No of the unit's active third party vendors who are 1. single source 
     vendors AND 2. located in high risk jurisdictions.
   - Rule 1: There is a file filtering single source with engagement ID. Then use 
     engagement ID to find 3rd party location and country risk rating.
   - Rule 2: Col D, M, N: any of these three columns have high risk country, counted as 1.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment 
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    -- Corrected Logic: The string 'Single Source' is in the EngagementNumber column.
    -- The actual E-Numbers (Engagement IDs) are stored in the ThirdPartyName column.
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        p.EngagementNumber,
        p.owning_lob,
        p.lob_using,
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    -- Bridge the real engagement ID to the main TP data table
    INNER JOIN Single_Source_Engagements s
        ON TRIM(p.EngagementNumber) = s.True_Engagement_ID
),

High_Risk_Engagements AS (
    -- Rule: Col D, M, N - any of these three columns have high risk country, counted as 1
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

Engagements_By_AU AS (
    SELECT 
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        COUNT(DISTINCT hr.EngagementNumber) AS High_Risk_Count
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping map
    INNER JOIN High_Risk_Engagements hr
        ON UPPER(hr.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
        OR UPPER(hr.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    WHERE map.Assessable_Unit_ID IS NOT NULL
    GROUP BY TRIM(CAST(map.Assessable_Unit_ID AS STRING))
)

SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP04' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.Base_AU_ID = e.Mapped_AU_ID
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP04 - TP Sole Source Drill-Down
   
   PURPOSE: Verifies the single source bridge was successful and shows the exact 
   country risk matching for Col D, M, and N.
=================================================================================== */

WITH Single_Source_Engagements AS (
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
)

SELECT DISTINCT
    map.Assessable_Unit_ID AS BusinessID,
    map.TPRM_Units AS Successfully_Mapped_String,
    tp.EngagementNumber,
    tp.owning_lob AS Raw_Col_K_owning_lob,
    tp.lob_using AS Raw_Col_L_lob_using,
    
    'Yes' AS Confirmed_Single_Source_Filter,
    
    tp.location_of_tp AS Col_D_Location,
    tp.country_prod_serv_originates AS Col_M_Origin,
    tp.Td_Countries_Impacted AS Col_N_Impacted,
    
    CASE 
        WHEN COALESCE(TRIM(tp.location_of_tp), '') = ''
         AND COALESCE(TRIM(tp.country_prod_serv_originates), '') = ''
         AND COALESCE(TRIM(tp.Td_Countries_Impacted), '') = '' THEN 'Missing Jurisdiction (Auto High Risk)'
        ELSE 'Matched High Risk ABAC List' 
    END AS Risk_Flag_Reason

FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp

-- Bridge to single source file to strictly keep single source engagements
INNER JOIN Single_Source_Engagements s
    ON TRIM(tp.EngagementNumber) = s.True_Engagement_ID

-- Bridge to TPRM Mapping
INNER JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    OR UPPER(tp.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'

-- Bridge to High Risk tables
LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName

-- Strictly filter for only the high-risk engagements
WHERE h1.CountryName IS NOT NULL
   OR h2.CountryName IS NOT NULL
   OR h3.CountryName IS NOT NULL
   OR (COALESCE(TRIM(tp.location_of_tp), '') = ''
       AND COALESCE(TRIM(tp.country_prod_serv_originates), '') = ''
       AND COALESCE(TRIM(tp.Td_Countries_Impacted), '') = '')
       
ORDER BY BusinessID;